In [ ]:
from pathlib import Path

# Find the repository root so these notebooks work on other computers.
PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "Dataset Specs").is_dir() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "Dataset Specs").is_dir():
    raise FileNotFoundError("Start Jupyter from inside the Hospital Dataset project folder.")

import pandas as pd
import numpy as np

file_path = str(PROJECT_ROOT / "Dataset Specs" / "hospital_occupancy_daily.csv")

df = pd.read_csv(file_path)
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").set_index("date")

daily_occupancy = df["occupied_beds"].asfreq("D")

print(daily_occupancy.head())

In [ ]:
import numpy as np

def make_lag_features(series, n_lags=7):
    df_ml = pd.DataFrame({"occupied_beds": series})
    
    #For Lag 1,2,...n
    for lag in range(1, n_lags + 1):
        df_ml[f"lag_{lag}"] = series.shift(lag)
    
    # Calendar features
    df_ml["day_of_week"] = df_ml.index.dayofweek  # 0=Mon ... 6=Sun
    df_ml["month"] = df_ml.index.month
    
    # Droprows that became NaN becuase of shifting
    df_ml = df_ml.dropna()
    return df_ml

ml_df = make_lag_features(daily_occupancy, n_lags=7)

print(ml_df.head())
print(ml_df.tail())

In [ ]:
data_ml = ml_df.sort_index()

train_size = int(len(data_ml) * 0.7)

train_ml = data_ml.iloc[:train_size]
test_ml  = data_ml.iloc[train_size:]

# Features (X) and Target (y)
feature_cols = ['lag_1', 'lag_7', 'day_of_week', 'month']

X_train = train_ml[feature_cols]
y_train = train_ml['occupied_beds']

X_test  = test_ml[feature_cols]
y_test  = test_ml['occupied_beds']

print("Train shape:", X_train.shape, "Test shape:", X_test.shape)
print("Train period:", X_train.index.min(), "to", X_train.index.max())
print("Test period :", X_test.index.min(), "to", X_test.index.max())

In [ ]:
# 70/30 split

data_ml = ml_df.dropna().sort_index()

train_size = int(len(data_ml) * 0.7)

train_ml = data_ml.iloc[:train_size]
test_ml  = data_ml.iloc[train_size:]

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error

def evaluate_model(y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    accuracy = 100 - mape
    return rmse, mae, mape, accuracy

In [ ]:
#Random Forest Model 1

from sklearn.ensemble import RandomForestRegressor

rf_model_1 = RandomForestRegressor(
    n_estimators=300,
    max_depth=None,
    random_state=42,
    n_jobs=-1
)

# Train_fit_model
rf_model_1.fit(X_train, y_train)
y_pred_rf_1 = rf_model_1.predict(X_test)
rmse1, mae1, mape1, acc1 = evaluate_model(y_test, y_pred_rf_1)

print("Random Forest model-1 trained.")

In [ ]:
#Random Forest Model 2

rf_model_2 = RandomForestRegressor(
    n_estimators=300,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)

rf_model_2.fit(X_train, y_train)
y_pred_rf_2 = rf_model_2.predict(X_test)
rmse2, mae2, mape2, acc2 = evaluate_model(y_test, y_pred_rf_2)

print("Random Forest model-2 trained.")

In [ ]:
#Random Forest Model 3

rf_model_3 = RandomForestRegressor(
    n_estimators=600,
    max_depth=None,
    random_state=42,
    n_jobs=-1
)

rf_model_3.fit(X_train, y_train)
y_pred_rf_3 = rf_model_3.predict(X_test)
rmse3, mae3, mape3, acc3 = evaluate_model(y_test, y_pred_rf_3)

print("Random Forest model-3 trained.")

In [ ]:
#Random Forest Model 4

rf_model_4 = RandomForestRegressor(
    n_estimators=300,
    max_depth=None,
    min_samples_split=10,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)

rf_model_4.fit(X_train, y_train)
y_pred_rf_4 = rf_model_4.predict(X_test)
rmse4, mae4, mape4, acc4 = evaluate_model(y_test, y_pred_rf_4)

print("Random Forest model-4 trained.")

In [ ]:
#Random Forest Model 5

rf_model_5 = RandomForestRegressor(
    n_estimators=500,
    max_depth=15,
    min_samples_leaf=3,
    random_state=42,
    n_jobs=-1
)

rf_model_5.fit(X_train, y_train)
y_pred_rf_5 = rf_model_5.predict(X_test)
rmse5, mae5, mape5, acc5 = evaluate_model(y_test, y_pred_rf_5)

print("Random Forest model-5 trained.")

In [ ]:
rf_pred_df = pd.DataFrame({
    "date": y_test.index,
    "rf_1": y_pred_rf_1,
    "rf_2": y_pred_rf_2,
    "rf_3": y_pred_rf_3,
    "rf_4": y_pred_rf_4,
    "rf_5": y_pred_rf_5
})

In [ ]:
rf_results = pd.DataFrame({
    "Model": ["RF-1", "RF-2", "RF-3", "RF-4", "RF-5"],
    "RMSE": [rmse1, rmse2, rmse3, rmse4, rmse5],
    "MAE":  [mae1,  mae2,  mae3,  mae4,  mae5],
    "MAPE (%)": [mape1, mape2, mape3, mape4, mape5],
    "Accuracy": [acc1, acc2, acc3, acc4, acc5]
})

rf_results

In [ ]:
rf1_series = pd.Series(y_pred_rf_1, index=y_test.index)
rf2_series = pd.Series(y_pred_rf_2, index=y_test.index)
rf3_series = pd.Series(y_pred_rf_3, index=y_test.index)
rf4_series = pd.Series(y_pred_rf_4, index=y_test.index)
rf5_series = pd.Series(y_pred_rf_5, index=y_test.index)

In [ ]:
#Plotting

import matplotlib.pyplot as plt

plt.figure(figsize=(14,6))

plt.plot(y_test.index, y_test, label="Actual", color="black", linewidth=2)

plt.plot(rf1_series, label="RF-1", alpha=0.6)
plt.plot(rf2_series, label="RF-2", alpha=0.6)
plt.plot(rf3_series, label="RF-3", alpha=0.6)
plt.plot(rf4_series, label="RF-4", alpha=0.6)
plt.plot(rf5_series, label="RF-5", alpha=0.6)

plt.title("Random Forest Variants: Actual vs Predictions")
plt.xlabel("Date")
plt.ylabel("Occupied Beds")
plt.legend()
plt.show()

In [ ]:
from pathlib import Path

output_dir = Path(
    f"{PROJECT_ROOT}/"
    "synthetic_hospital_outputs/70_30"
)
output_dir.mkdir(parents=True, exist_ok=True)

rf_pred_df.to_csv(
    output_dir / "random_forest_occupancy_predictions.csv",
    index=False
)

print(f"Random Forest predictions saved to: {output_dir}")